# v17c — trained baselines on MSLSAM's exact folds

v17 run 1 showed MSLSAM **in-distribution** leading on SPML, DeepSet and BIPIA — dramatically on
BIPIA (36% ASR vs ~89% for every other method). But MSLSAM was the **only** method trained on
this corpus, so that lead cannot yet be credited to Llama's hidden states rather than to simply
having seen the data. This notebook removes that confound.

## Two trained baselines, each answering a different question

| baseline | what it is | the question it answers |
|---|---|---|
| **TF-IDF + logistic regression** | word 1–2-grams + char 3–5-grams, the classical text classifier | *is the lead just in-domain training?* |
| **MiniLM embedding probe** | frozen `all-MiniLM-L6-v2` (22M, general-purpose), mean-pooled, then a head | *is it Llama's hidden states specifically, or would any learned encoder do?* |

The MiniLM probe is the closest structural twin of MSLSAM: frozen representation, trained head.
It gets **both** a logistic-regression head and **the same HistGradientBoost head MSLSAM uses**,
because v14's central lesson was that the head, not the features, was the bottleneck — denying the
baseline MSLSAM's head would rig the comparison.

## Identical folds — asserted, not assumed

Same rows, same labels, same `group_carrier` groups, same `StratifiedGroupKFold(5, shuffle=True,
random_state=42)` call as v17b's MSLSAM, and the same four leave-one-corpus-out holdouts. The fold
assignment is saved to `scores/_folds_indist.npy`, and **v17b refuses to compare if its own MSLSAM
folds differ.** Every vectoriser and scaler is fitted on the training fold only.

## Pre-registered reading (written before the run)

- **TF-IDF in-dist within 2 pooled-ASR points of MSLSAM in-dist (14.59%), or better** → MSLSAM's
  in-distribution lead is the **training data**, not the hidden states. Say so plainly.
- **MSLSAM ahead by more** → a real representational advantage *candidate*; it becomes a claim
  only family by family, via the paired McNemar test v17b now runs on identical injections.
- **MiniLM probe ≈ MSLSAM** → the advantage belongs to "a learned representation", not to Llama.
- **BIPIA is the family to watch.** It is where MSLSAM's lead is largest and where word n-grams
  should struggle most, because the payload is diluted inside a long carrier document.

## Cost
TF-IDF is CPU-only and fast. MiniLM embeds 11,245 rows once on MPS (a few minutes), then both
heads train on CPU. Batch-size-1 latency is timed warm for each, like every other method.

In [1]:
import os, json, time, gc
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

SEED, N_SPLITS, LAT_N = 42, 5, 50
SC_DIR = "results/v17/scores"
os.makedirs(SC_DIR, exist_ok=True)

corpus = pd.read_csv("datasets/unified/injection_corpus_v15.csv")
df = corpus[corpus.corpus != "Baseline-Dolly"].reset_index(drop=True)
rows = pd.read_csv("representation_cache/joined_v16/rows_v16.csv")
ev = rows[rows.corpus != "Baseline-Dolly"].reset_index(drop=True)      # exactly v17b's `ev`
assert len(ev) == len(df), "row count differs from v17b"
assert (ev.corpus.values == df.corpus.values).all() and (ev.label.values == df.label.values).all(), \
       "row order differs from v17b -- folds would not match"
r = pd.read_csv(f"{SC_DIR}/_rows.csv")
assert (r.corpus.values == df.corpus.values).all(), "scores directory is aligned to another order"

texts = df.text.astype(str).tolist()
y, groups, cor = ev.label.values, ev.group_carrier.values, ev.corpus.values

# IDENTICAL to v17b: StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED)
#                    .split(np.zeros(len(ev)), ev.label.values, groups=ev["group_carrier"].values)
skf = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = [(np.asarray(tr), np.asarray(te))
         for tr, te in skf.split(np.zeros(len(ev)), y, groups=groups)]
fold_id = np.full(len(ev), -1)
for k, (_, te) in enumerate(FOLDS): fold_id[te] = k
assert (fold_id >= 0).all()
np.save(f"{SC_DIR}/_folds_indist.npy", fold_id)

LOCO_FAM = {"SPML": ["SPML", "SPML-Benign"], "DeepSet": ["DeepSet", "DeepSet-Benign"],
            "BIPIA": ["BIPIA", "BIPIA-Clean", "BIPIA-HardNeg"],
            "InjecAgent": ["InjecAgent", "InjecAgent-Clean", "InjecAgent-HardNeg"]}

def run_protocol(fit_predict, name):
    # fit_predict(train_idx, test_idx) -> scores for test_idx. Everything that learns
    # (vocabulary, scaler, head) must be fitted INSIDE it, on train_idx only.
    t0 = time.perf_counter()
    indist = np.full(len(ev), np.nan)
    for k, (tr, te) in enumerate(FOLDS):
        indist[te] = fit_predict(tr, te)
    loco = np.full(len(ev), np.nan)
    for fam, cs in LOCO_FAM.items():
        hold = np.isin(cor, cs)
        loco[hold] = fit_predict(np.where(~hold)[0], np.where(hold)[0])
    # In-dist must be complete. LOCO must be complete on the 4 held-out families only:
    # WJB-Adv-Benign / WJB-Vanilla-Benign / XSTest-Safe / JBB-Benign belong to no family, are
    # never held out, and stay NaN BY DESIGN -- exactly like MSLSAM's LOCO array in v17b,
    # whose nested() drops NaNs. (Run 1 asserted the whole array and crashed here; the dry
    # run missed it because its synthetic corpus had no non-family corpora.)
    fam_rows = np.isin(cor, sum(LOCO_FAM.values(), []))
    assert not np.isnan(indist).any(), f"{name}: gaps in in-dist scores"
    assert not np.isnan(loco[fam_rows]).any(), f"{name}: gaps in LOCO scores on held-out families"
    assert np.isnan(loco[~fam_rows]).all(), f"{name}: LOCO scored rows outside the 4 families"
    dt = time.perf_counter() - t0
    auc = roc_auc_score(y, indist)
    print(f"  {name:<14} in-dist AUC {auc:.4f} | 9 fits in {dt:.1f}s")
    return indist, loco, dt

def save(name, arr):
    arr = np.asarray(arr, np.float32); assert len(arr) == len(df)
    np.save(f"{SC_DIR}/{name}.npy", arr); print(f"    saved {name}.npy")

print(f"{len(ev):,} rows | {int(y.sum()):,} injections | {len(set(groups)):,} carrier groups")
print(f"folds saved -> {SC_DIR}/_folds_indist.npy (v17b will assert its MSLSAM folds match)")

11,245 rows | 4,148 injections | 6,042 carrier groups
folds saved -> results/v17/scores/_folds_indist.npy (v17b will assert its MSLSAM folds match)


In [2]:
# ================= TF-IDF + logistic regression ==============================
# Fixed, conventional settings -- NOT tuned on this corpus (a tuned baseline and an untuned
# MSLSAM would be as unfair as the reverse). Word 1-2-grams catch phrasing ("ignore previous");
# char_wb 3-5-grams catch obfuscation and German inflection in DeepSet.
def make_tfidf():
    return make_pipeline(
        make_union(
            TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                            max_features=200_000, sublinear_tf=True),
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                            max_features=300_000, sublinear_tf=True)),
        LogisticRegression(C=4.0, class_weight="balanced", max_iter=3000,
                           solver="liblinear", random_state=SEED))

TX = np.array(texts, dtype=object)
def tfidf_fp(tr, te):
    m = make_tfidf().fit(TX[tr], y[tr])
    return m.predict_proba(TX[te])[:, 1]

tf_in, tf_loco, tf_dt = run_protocol(tfidf_fp, "TF-IDF + LR")
save("tfidf_lr_indist", tf_in); save("tfidf_lr_loco", tf_loco)

# latency: one model fitted on everything, single prompts, warm (first call untimed)
full = make_tfidf().fit(TX, y)
full.predict_proba(TX[:1])
lat = []
for t in texts[:LAT_N]:
    a = time.perf_counter(); full.predict_proba(np.array([t], dtype=object)); lat.append(time.perf_counter() - a)
json.dump({"ms_per_prompt": float(np.median(lat)*1e3), "latency_bs1_ms": float(np.median(lat)*1e3),
           "params": int(sum(len(v.vocabulary_) for _, v in full[0].transformer_list)) + 1,
           "needs_forward_pass": False, "train_seconds_9_fits": tf_dt,
           "config": "word 1-2g + char_wb 3-5g TF-IDF, LR C=4 balanced, untuned"},
          open(f"{SC_DIR}/tfidf_lr.cost.json", "w"), indent=2)
print(f"  latency (bs=1, warm) {np.median(lat)*1e3:.3f} ms/prompt")
del full; gc.collect()

  TF-IDF + LR    in-dist AUC 0.9883 | 9 fits in 45.2s
    saved tfidf_lr_indist.npy
    saved tfidf_lr_loco.npy
  latency (bs=1, warm) 0.642 ms/prompt


509

In [3]:
# ================= MiniLM embedding probe ===================================
# A general-purpose sentence encoder, frozen, then a trained head -- MSLSAM's structural twin.
# max_length 512 on purpose: BIPIA payloads often sit at the END of long carriers, and the
# common 256-token setting would truncate them away and handicap the baseline unfairly.
import torch
from transformers import AutoTokenizer, AutoModel
DEV = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
ENC_ID, ENC_MAX, PAD_MULT, BS = "sentence-transformers/all-MiniLM-L6-v2", 512, 64, 64
EMB_PATH = "results/v17/minilm_emb.npy"

etok = AutoTokenizer.from_pretrained(ENC_ID)
enc = AutoModel.from_pretrained(ENC_ID).to(DEV).eval()
n_enc = sum(p.numel() for p in enc.parameters())
print(f"{ENC_ID} on {DEV} | {n_enc/1e6:.1f}M params")

def embed(batch):
    x = etok(batch, padding=True, truncation=True, max_length=ENC_MAX,
             pad_to_multiple_of=PAD_MULT, return_tensors="pt").to(DEV)
    with torch.inference_mode():
        h = enc(**x).last_hidden_state
        m = x["attention_mask"].unsqueeze(-1).to(h.dtype)
        e = (h * m).sum(1) / m.sum(1).clamp(min=1)          # mean pooling, the model's intended use
        e = torch.nn.functional.normalize(e, dim=-1)
    return e.float().cpu().numpy()

if os.path.exists(EMB_PATH):
    EMB = np.load(EMB_PATH); assert EMB.shape[0] == len(texts); print("loaded cached embeddings")
else:
    embed(texts[:2])                                        # warm-up
    order = np.argsort([len(t) for t in texts])             # length-sorted: little padding
    EMB = np.zeros((len(texts), enc.config.hidden_size), np.float32)
    t0 = time.perf_counter()
    for i in range(0, len(order), BS):
        ix = order[i:i+BS]; EMB[ix] = embed([texts[j] for j in ix])
    print(f"embedded {len(texts):,} rows in {time.perf_counter()-t0:.0f}s")
    np.save(EMB_PATH, EMB)

lat = []
for t in texts[:LAT_N]:
    embed([t])
    if DEV == "mps": torch.mps.synchronize()
    a = time.perf_counter(); embed([t])
    if DEV == "mps": torch.mps.synchronize()
    lat.append(time.perf_counter() - a)
enc_lat = float(np.median(lat) * 1e3)
print(f"encoder latency (bs=1, warm) {enc_lat:.1f} ms")
del enc; gc.collect()
if DEV == "mps": torch.mps.empty_cache()

def lr_fp(tr, te):
    sc = StandardScaler().fit(EMB[tr])
    c = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000,
                           random_state=SEED).fit(sc.transform(EMB[tr]), y[tr])
    return c.predict_proba(sc.transform(EMB[te]))[:, 1]

def hgb_fp(tr, te):                                          # MSLSAM's own head, same settings
    c = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(EMB[tr], y[tr])
    return c.predict_proba(EMB[te])[:, 1]

ml_in, ml_loco, ml_dt = run_protocol(lr_fp,  "MiniLM + LR")
save("minilm_lr_indist", ml_in); save("minilm_lr_loco", ml_loco)
mh_in, mh_loco, mh_dt = run_protocol(hgb_fp, "MiniLM + HGB")
save("minilm_hgb_indist", mh_in); save("minilm_hgb_loco", mh_loco)

json.dump({"ms_per_prompt": enc_lat, "latency_bs1_ms": enc_lat, "params": int(n_enc),
           "needs_forward_pass": True, "hf_id": ENC_ID, "max_length": ENC_MAX,
           "note": "latency is the frozen encoder; head inference is < 1 ms",
           "train_seconds_9_fits_lr": ml_dt, "train_seconds_9_fits_hgb": mh_dt},
          open(f"{SC_DIR}/minilm_probe.cost.json", "w"), indent=2)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/all-MiniLM-L6-v2 on mps | 22.7M params
embedded 11,245 rows in 68s
encoder latency (bs=1, warm) 8.8 ms


/Users/stanleypratamateguh/miniconda3/envs/llm_eval/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/stanleypratamateguh/miniconda3/envs/llm_eval/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/stanleypratamateguh/miniconda3/envs/llm_eval/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/stanleypratamateguh/miniconda3/envs/llm_eval/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/stanleypratamateguh/miniconda3/envs/llm_eval/lib/python3.10/site-packages/sklearn/linear_model/_

  MiniLM + LR    in-dist AUC 0.9424 | 9 fits in 1.4s
    saved minilm_lr_indist.npy
    saved minilm_lr_loco.npy
  MiniLM + HGB   in-dist AUC 0.9590 | 9 fits in 33.0s
    saved minilm_hgb_indist.npy
    saved minilm_hgb_loco.npy


In [4]:
# ---- quick sanity view (the real comparison is v17b's) ----------------------
print(f"{'family':<12}{'TF-IDF':>10}{'MiniLM-LR':>11}{'MiniLM-HGB':>12}   (in-dist ROC-AUC vs own benign)")
PAIR = {"SPML": "SPML-Benign", "DeepSet": "DeepSet-Benign",
        "BIPIA": "BIPIA-Clean", "InjecAgent": "InjecAgent-Clean"}
for f, b in PAIR.items():
    m = np.isin(cor, [f, b]); yy = (cor[m] == f).astype(int)
    print(f"{f:<12}" + "".join(f"{roc_auc_score(yy, s[m]):>{w}.4f}"
                               for s, w in ((tf_in, 10), (ml_in, 11), (mh_in, 12))))
print("\non disk:", sorted(f for f in os.listdir(SC_DIR) if not f.startswith("_")))
print("\nNEXT: v17b_compare.ipynb -- it picks these up automatically and runs the paired test.")

family          TF-IDF  MiniLM-LR  MiniLM-HGB   (in-dist ROC-AUC vs own benign)
SPML            0.9953     0.9813      0.9943
DeepSet         0.9043     0.7995      0.8533
BIPIA           0.9474     0.7088      0.7965
InjecAgent      0.9995     0.7947      0.7781

on disk: ['llm_judge.cost.json', 'llm_judge.npy', 'minilm_hgb_indist.npy', 'minilm_hgb_loco.npy', 'minilm_lr_indist.npy', 'minilm_lr_loco.npy', 'minilm_probe.cost.json', 'promptguard2_86m.cost.json', 'promptguard2_86m.npy', 'protectai_deberta.cost.json', 'protectai_deberta.npy', 'rule_binary.npy', 'rule_count.npy', 'rule_keyword.cost.json', 'tfidf_lr.cost.json', 'tfidf_lr_indist.npy', 'tfidf_lr_loco.npy']

NEXT: v17b_compare.ipynb -- it picks these up automatically and runs the paired test.
